In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
import torch.nn.functional as F 
import os

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [30]:
def predict_all_images(Model, directory_path='.'):
    # 1. Definition of transformations
    transform = transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 2. Image file restriction
    valid_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tif')
    image_files = [f for f in os.listdir(directory_path) 
                   if f.lower().endswith(valid_extensions)]

    if not image_files:
        print(f"❌ لم يتم العثور على صور في المجلد: {directory_path}")
        return

    class_names = ['drink', 'food', 'interior', 'menu', 'outside']
    stats = {name: 0 for name in class_names}  # قاموس للعد
    errors = 0

    print(f"تم العثور على {len(image_files)} صور. جاري التنبؤ...\n")
    print(f"{'Image Name':<40} | {'Prediction':<15} | {'Confidence'}")
    print("-" * 75)

    Model.eval()
    with torch.no_grad():
        for img_name in image_files:
            try:
                img_path = os.path.join(directory_path, img_name)
                img = Image.open(img_path).convert('RGB')
                img_tensor = transform(img).unsqueeze(0).to(device)

                # Prediction
                outputs = Model(img_tensor)
                probabilities = F.softmax(outputs, dim=1)
                confidence, preds = torch.max(probabilities, 1)

                label = class_names[preds.item()]
                score = confidence.item() * 100

                # Statistics update
                stats[label] += 1

                print(f"{img_name:<40} | {label:<15} | {score:.2f}%")
            
            except Exception as e:
                errors += 1
                print(f"{img_name:<40} | ❌ Error: {str(e)}")

    print("\n" + "="*40)
    print("📊 الإحصائية النهائية للتصنيف:")
    print("="*40)
    for category, count in stats.items():
        percentage = (count / len(image_files)) * 100 if len(image_files) > 0 else 0
        print(f"🔹 {category:<10}: {count:>5} صور ({percentage:.1f}%)")
    
    if errors > 0:
        print(f"⚠️ أخطاء برمجية: {errors} ملفات")
    print("="*40)

# استدعاء الدالة
# predict_all_images(model, directory_path="E:/Dataset/Test")

In [11]:
# 1. Settings and paths
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data_dir = 'Dataset'
PATH = r"Enter the path to the model after downloading it from GitHub or anywhere" 

# 2. Data Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((448, 448)), 
        transforms.Grayscale(num_output_channels=3), 
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomRotation(15), 
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((448, 448)),
        transforms.Grayscale(num_output_channels=3), 
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [7]:
# 3. Download data directly from pre made folders

train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), data_transforms['train'])
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, 'valid'),   data_transforms['test'])
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, 'test'),  data_transforms['test'])

# Dataloaders for each group
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=16, shuffle=True),
    'valid':   DataLoader(val_dataset,   batch_size=16, shuffle=False),
    'test':  DataLoader(test_dataset,  batch_size=16, shuffle=False)
}

# Print data sizes
print(f"✅ التدريب: {len(train_dataset)} صورة")
print(f"✅ التحقق: {len(val_dataset)} صورة")
print(f"✅ الاختبار: {len(test_dataset)} صورة")

✅ التدريب: 6000 صورة
✅ التحقق: 1000 صورة
✅ الاختبار: 1000 صورة


In [9]:
# 4. Preparing the model from the local file

model = models.resnet50(weights=None) 

state_dict = torch.load(PATH, map_location=device)
model.load_state_dict(state_dict)
print("✅ Local weights loaded successfully.")
# Freeze the model
for param in model.parameters():
    param.requires_grad = False
# Model head modification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5) 
# Data transfer to the GPU
model = model.to(device)

✅ Local weights loaded successfully.


In [11]:
# 5. Preparing for training
# Loss Function
criterion = nn.CrossEntropyLoss()
# Optimizer
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

In [15]:
# 6. Training Loop
print("🚀 Starting Training...")
num_epochs = 20
patience = 5 
best_val_loss = float('inf')
counter = 0

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}:")
    # Training phase
    model.train()
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1) 
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
    
    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_acc = running_corrects.double() / len(train_dataset)

    # Verification phase
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloaders['valid']:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_corrects.double() / len(val_dataset)
    
    print(f"  Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4%}")
    print(f"  Val Loss:   {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4%}")

    # Stop early and save the best (based on Loss)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_resnet50_menu_classification.pth')
        print("⭐ Improvement found! Best model saved.")
    else:
        counter += 1
        print(f"🐢 No improvement for {counter} epochs.")
        
    if counter >= patience:
        print("🛑 Early stopping triggered.")
        break

print("✅ Training complete.")

🚀 Starting Training...
Epoch 1/20:


C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


  Train Loss: 0.5397 Acc: 83.0500%
  Val Loss:   0.3775 Acc: 87.7000%
⭐ Improvement found! Best model saved.
Epoch 2/20:
  Train Loss: 0.3608 Acc: 87.8667%
  Val Loss:   0.3244 Acc: 90.1000%
⭐ Improvement found! Best model saved.
Epoch 3/20:
  Train Loss: 0.3368 Acc: 88.0667%
  Val Loss:   0.2788 Acc: 90.7000%
⭐ Improvement found! Best model saved.
Epoch 4/20:
  Train Loss: 0.3130 Acc: 89.1833%
  Val Loss:   0.2714 Acc: 90.4000%
⭐ Improvement found! Best model saved.
Epoch 5/20:
  Train Loss: 0.2975 Acc: 89.7000%
  Val Loss:   0.2649 Acc: 91.0000%
⭐ Improvement found! Best model saved.
Epoch 6/20:
  Train Loss: 0.2837 Acc: 89.6833%
  Val Loss:   0.2650 Acc: 90.9000%
🐢 No improvement for 1 epochs.
Epoch 7/20:
  Train Loss: 0.2781 Acc: 89.9167%
  Val Loss:   0.2501 Acc: 91.0000%
⭐ Improvement found! Best model saved.
Epoch 8/20:
  Train Loss: 0.2629 Acc: 90.6833%
  Val Loss:   0.2456 Acc: 91.3000%
⭐ Improvement found! Best model saved.
Epoch 9/20:
  Train Loss: 0.2689 Acc: 89.9333%
  Val

In [21]:
# 7. Final evaluation using the best model

# 7.1. Download the best saved weights
model.load_state_dict(torch.load('best_resnet50_menu_classification.pth'))
model.eval()

def evaluate_set(loader, name):
    running_loss = 0.0
    running_corrects = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
    total_size = len(loader.dataset)
    avg_loss = running_loss / total_size
    accuracy = running_corrects.double() / total_size
    
    print(f"📊 {name:10} -> Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2%}")
    return avg_loss, accuracy

print("\n🧐 Final Evaluation on All Sets:")
print("-" * 45)

train_loss, train_acc = evaluate_set(dataloaders['train'], "Training")
val_loss, val_acc     = evaluate_set(dataloaders['valid'],   "Validation")
test_loss, test_acc   = evaluate_set(dataloaders['test'],  "Testing")

print("-" * 45)
# Train:Loss:0.1872|Accuracy:93.77% >> drink:94.8, food:93.1, interior:89.0, menu:97.8, outside:92.2
# valid:Loss0.2409|Accuracy:91.60% >> drink:90, food:96.4, interior:90.0, menu:96.5, outside:87.9
# test:Loss:0.2281|Accuracy:92.20% >> drink:93, food:94.6, interior:90.0, menu:96, outside:87.4



🧐 Final Evaluation on All Sets:
---------------------------------------------
📊 Training   -> Loss: 0.1872 | Accuracy: 93.77%
📊 Validation -> Loss: 0.2409 | Accuracy: 91.60%
📊 Testing    -> Loss: 0.2281 | Accuracy: 92.20%
---------------------------------------------


In [39]:
model.eval()
all_preds = []
all_labels = []

with torch.inference_mode():
    for X, y in dataloaders['test']:   # ضع داتالودر التحقق
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())


C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return F.conv2d(input, weight, bias, self.stride,


In [41]:
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)


Precision: 0.923228188594645
Recall: 0.921856385638564
F1 Score: 0.9221614206087596


In [53]:
cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[186   4   5   4   1]
 [  3 191   5   0   3]
 [  3   6 180   3   8]
 [  1   0   5 192   2]
 [  2   1  19   3 173]]


In [55]:
print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       200
           1       0.95      0.95      0.95       202
           2       0.84      0.90      0.87       200
           3       0.95      0.96      0.96       200
           4       0.93      0.87      0.90       198

    accuracy                           0.92      1000
   macro avg       0.92      0.92      0.92      1000
weighted avg       0.92      0.92      0.92      1000



In [61]:
# drink  food interior menu outside
directory_path= r'r"Deep-Learning-Projects\Menu_Classification\Dataset\test\outside'
predict_all_images(model, directory_path)

🚀 تم العثور على 198 صور. جاري التنبؤ...

Image Name                               | Prediction      | Confidence
---------------------------------------------------------------------------
0Dp3vjwSgf5oUtod-zAf2A.jpg               | outside         | 99.94%
0Mt_83-FZMxzhM_2oENqUA.jpg               | outside         | 96.54%
1FohJHRlFJ5edxqzAjvAbQ.jpg               | outside         | 99.61%
1k0IBLvQHxLleD6I_KLBYQ.jpg               | outside         | 99.79%
2LJEUjtYtm0__m8Pr5vwNQ.jpg               | outside         | 93.50%
2mPnV3gVbMobDjKPPuFHQg.jpg               | outside         | 59.02%
31_XckPXD0noDfpk7YHcQg.jpg               | outside         | 90.93%
3DRTagJG7vRYSRf2NHYLRA.jpg               | outside         | 99.77%
3et3xyiAA_bDV1okcdtvCw.jpg               | outside         | 100.00%
49vmcZKRSLFnhMnTBivw4g.jpg               | outside         | 99.99%
4CWQorkDq7AIshsq3dJURA.jpg               | interior        | 57.34%
4JgBG-0VBodiTKtvc3ldEA.jpg               | outside         | 9